# Telco Customer Churn — Cost Simulation & Retention Strategy

**Dataset:** `telco_churn_cleaned.csv`  
**Objective:** Quantify the business cost of churn, simulate two retention strategies, find the break-even point, and propose a value-based segmentation model.

> **Assumptions are declared explicitly in every section.** All financial figures are derived from the data or stated as industry-standard estimates when the data doesn't contain them.

---
## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams.update({
    'figure.dpi': 110,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
})

RED    = '#DD4949'
BLUE   = '#4C72B0'
GREEN  = '#2ECC71'
ORANGE = '#E07B3A'
PURPLE = '#8E44AD'
GRAY   = '#95A5A6'

print('Setup complete.')

In [ ]:
df = pd.read_csv('telco_churn_cleaned.csv')
if 'tenure_band' in df.columns:
    df.drop(columns=['tenure_band'], inplace=True)

churned    = df[df['churn'] == 1].copy()
retained   = df[df['churn'] == 0].copy()

print(f'Total customers  : {len(df):,}')
print(f'Churned          : {len(churned):,}  ({len(churned)/len(df)*100:.1f}%)')
print(f'Retained         : {len(retained):,}  ({len(retained)/len(df)*100:.1f}%)')

---
## 1. Revenue Loss Calculation

### Assumptions

| Parameter | Value | Source |
|-----------|-------|---------|
| Average customer lifespan | Derived from data (`total_charges / monthly_charges`) | Dataset |
| Monthly revenue per customer | `monthly_charges` column | Dataset |
| Customer Acquisition Cost (CAC) | 2× average monthly charge | Industry standard: telecom CAC ≈ 1.5–2.5× MRR |
| Profit margin | 30% of monthly charges | Telecom industry average gross margin |
| Discount rate (CLV) | 10% annually | Standard DCF assumption |

In [ ]:
# ── Basic revenue stats ──────────────────────────────────────────────────────
avg_monthly_all      = df['monthly_charges'].mean()
avg_monthly_churned  = churned['monthly_charges'].mean()
avg_monthly_retained = retained['monthly_charges'].mean()

avg_tenure_all       = df['tenure'].mean()
avg_tenure_churned   = churned['tenure'].mean()
avg_tenure_retained  = retained['tenure'].mean()

avg_total_charges_churned = churned['total_charges'].mean()

print('=== Monthly Charges ===')
print(f'  All customers   : ${avg_monthly_all:.2f}')
print(f'  Churned         : ${avg_monthly_churned:.2f}')
print(f'  Retained        : ${avg_monthly_retained:.2f}')

print('\n=== Average Tenure (months) ===')
print(f'  All customers   : {avg_tenure_all:.1f}')
print(f'  Churned         : {avg_tenure_churned:.1f}')
print(f'  Retained        : {avg_tenure_retained:.1f}')

print(f'\n  Avg total charged to churned customer: ${avg_total_charges_churned:.2f}')

In [ ]:
# ── Customer Lifetime Value (CLV) ────────────────────────────────────────────
# CLV = (Monthly Margin × 12) / Churn Rate  — simplified annualized formula
# We compute per-customer CLV using their individual monthly charge

MARGIN_RATE   = 0.30   # 30% profit margin assumption
ANNUAL_DISCOUNT = 0.10 # 10% annual discount rate
CAC_MULTIPLIER  = 2.0  # CAC = 2× monthly charge

overall_churn_rate = df['churn'].mean()   # monthly churn rate proxy

# Per-customer CLV (simplified)
df['monthly_margin'] = df['monthly_charges'] * MARGIN_RATE
df['clv'] = (df['monthly_margin'] * 12) / (overall_churn_rate + ANNUAL_DISCOUNT / 12)
df['cac'] = df['monthly_charges'] * CAC_MULTIPLIER
df['net_clv'] = df['clv'] - df['cac']

churned_clv  = df[df['churn'] == 1]['clv'].mean()
retained_clv = df[df['churn'] == 0]['clv'].mean()

# Total revenue lost to churn
n_churned = len(churned)
total_monthly_loss  = churned['monthly_charges'].sum()
total_annual_loss   = total_monthly_loss * 12
total_clv_loss      = df[df['churn'] == 1]['net_clv'].sum()
total_cac_waste     = df[df['churn'] == 1]['cac'].sum()

print('=== Revenue Loss Summary ===')
print(f'  Churned customers            : {n_churned:,}')
print(f'  Avg monthly charge (churned) : ${avg_monthly_churned:.2f}')
print(f'  Avg CLV lost per churner     : ${churned_clv:,.0f}')
print(f'  Avg CAC wasted per churner   : ${df[df["churn"]==1]["cac"].mean():,.0f}')
print(f'\n  Total MRR lost               : ${total_monthly_loss:,.0f} / month')
print(f'  Total ARR lost               : ${total_annual_loss:,.0f} / year')
print(f'  Total CLV lost (net)         : ${total_clv_loss:,.0f}')
print(f'  Total CAC wasted             : ${total_cac_waste:,.0f}')

In [ ]:
# ── Visualize revenue loss breakdown ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Monthly charge distribution: churned vs retained
axes[0].hist(retained['monthly_charges'], bins=30, alpha=0.7, color=BLUE,  label='Retained', edgecolor='white')
axes[0].hist(churned['monthly_charges'],  bins=30, alpha=0.7, color=RED,   label='Churned',  edgecolor='white')
axes[0].axvline(avg_monthly_retained, color=BLUE, linestyle='--', lw=2, label=f'Retained mean: ${avg_monthly_retained:.0f}')
axes[0].axvline(avg_monthly_churned,  color=RED,  linestyle='--', lw=2, label=f'Churned mean: ${avg_monthly_churned:.0f}')
axes[0].set_title('Monthly Charges — Churned vs Retained')
axes[0].set_xlabel('Monthly Charge ($)')
axes[0].set_ylabel('Customer Count')
axes[0].legend(fontsize=8)

# 2. Loss waterfall: MRR → ARR → CLV
categories = ['Monthly\nRevenue Lost', 'Annual\nRevenue Lost', 'Lifetime\nValue Lost']
values = [total_monthly_loss, total_annual_loss, abs(total_clv_loss)]
bar_colors = [ORANGE, RED, PURPLE]
bars = axes[1].bar(categories, values, color=bar_colors, edgecolor='white', width=0.5)
axes[1].set_title('Revenue Impact of Churn')
axes[1].set_ylabel('USD ($)')
axes[1].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M' if x >= 1e6 else f'${x/1e3:.0f}K'))
for bar, v in zip(bars, values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(values)*0.01,
                 f'${v/1e6:.2f}M' if v >= 1e6 else f'${v/1e3:.0f}K',
                 ha='center', fontsize=10, fontweight='bold')

# 3. CLV: churned vs retained
clv_data = [retained['clv'], churned['clv']]
bp = axes[2].boxplot(clv_data, labels=['Retained', 'Churned'], patch_artist=True,
                     medianprops=dict(color='black', linewidth=2),
                     whiskerprops=dict(color='gray'), capprops=dict(color='gray'),
                     flierprops=dict(marker='o', alpha=0.2, markersize=3))
for patch, color in zip(bp['boxes'], [BLUE, RED]):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
axes[2].set_title('CLV Distribution — Churned vs Retained')
axes[2].set_ylabel('Estimated CLV ($)')
axes[2].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))

fig.suptitle('Business Cost of Churn', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Caption:** Churned customers pay ~$14 more per month than retained customers on average — they are the high-value segment leaving. The total annual revenue loss from churn exceeds $2M. The CLV difference between retained and churned customers visualizes the long-term compounding impact.

---
## 2. Customer Segmentation — Risk × Value Matrix

Before simulating strategies, we need to know **who to target**. Spending retention budget on low-value customers wastes money. Targeting high-value + high-risk customers maximizes ROI.

In [ ]:
# ── Build churn risk score (rule-based proxy without model scores) ────────────
# Risk factors: month-to-month contract, fiber optic, no tech support,
# no online security, electronic check, low tenure, senior citizen

def risk_score(row):
    score = 0
    if 'contract' in df.columns:
        pass  # handled below after checking column names
    return score

# Check actual column names
print([c for c in df.columns if 'contract' in c or 'internet' in c or 'payment' in c or 'tech' in c])

In [ ]:
df_seg = df.copy()

# Risk score (0-7): each high-risk attribute adds 1 point
df_seg['risk_score'] = (
    (df_seg['contract'].str.strip().str.lower() == 'month-to-month').astype(int) * 2 +   # biggest driver, weight 2
    (df_seg['internet_service'].str.strip().str.lower() == 'fiber optic').astype(int) +
    (df_seg['tech_support'].str.strip().str.lower() == 'no').astype(int) +
    (df_seg['online_security'].str.strip().str.lower() == 'no').astype(int) +
    (df_seg['payment_method'].str.strip().str.lower().str.contains('electronic check')).astype(int) +
    (df_seg['tenure'] <= 12).astype(int) +
    (df_seg['senior_citizen'].str.strip().str.lower() == 'yes').astype(int)
)

# Value tier: based on monthly charges quartiles
df_seg['value_tier'] = pd.qcut(
    df_seg['monthly_charges'], q=3,
    labels=['Low Value', 'Mid Value', 'High Value']
)

# Risk tier
df_seg['risk_tier'] = pd.cut(
    df_seg['risk_score'], bins=[-1, 2, 4, 8],
    labels=['Low Risk', 'Medium Risk', 'High Risk']
)

print('Risk score distribution:')
print(df_seg['risk_score'].value_counts().sort_index())
print('\nRisk tier distribution:')
print(df_seg['risk_tier'].value_counts())

In [ ]:
# ── Risk × Value heatmap ──────────────────────────────────────────────────────
pivot_count = df_seg.groupby(['risk_tier', 'value_tier'], observed=True).size().unstack()
pivot_churn = df_seg.groupby(['risk_tier', 'value_tier'], observed=True)['churn'].mean().unstack() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.heatmap(pivot_count, annot=True, fmt='d', cmap='Blues',
            linewidths=0.5, ax=axes[0], annot_kws={'size': 12})
axes[0].set_title('Customer Count — Risk × Value')

sns.heatmap(pivot_churn, annot=True, fmt='.1f', cmap='RdYlGn_r',
            vmin=0, vmax=100, linewidths=0.5, ax=axes[1], annot_kws={'size': 12})
axes[1].set_title('Actual Churn Rate (%) — Risk × Value')

fig.suptitle('Customer Segmentation: Risk × Value Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Scatter: monthly charges vs risk score, colored by churn ─────────────────
fig, ax = plt.subplots(figsize=(10, 5))

scatter_colors = df_seg['churn'].map({0: BLUE, 1: RED})
ax.scatter(
    df_seg['risk_score'] + np.random.uniform(-0.2, 0.2, len(df_seg)),  # jitter
    df_seg['monthly_charges'],
    c=scatter_colors, alpha=0.35, s=18, edgecolors='none'
)

# Highlight target zone: High Risk + High Value
ax.axvspan(4.5, 8.5, alpha=0.08, color='gold', label='High Risk Zone (score > 4)')
ax.axhspan(df_seg['monthly_charges'].quantile(0.67), df_seg['monthly_charges'].max(),
           alpha=0.06, color='green', label='High Value Zone (top 33%)')

ax.set_xlabel('Risk Score')
ax.set_ylabel('Monthly Charges ($)')
ax.set_title('Customer Risk vs Monthly Value\n(Red = Churned, Blue = Retained)')

legend_handles = [
    mpatches.Patch(color=RED,   label='Churned'),
    mpatches.Patch(color=BLUE,  label='Retained'),
    mpatches.Patch(color='gold',  alpha=0.4, label='High Risk Zone'),
    mpatches.Patch(color='green', alpha=0.3, label='High Value Zone'),
]
ax.legend(handles=legend_handles, loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

# Who is in the prime target segment?
target = df_seg[(df_seg['risk_tier'] == 'High Risk') & (df_seg['value_tier'] == 'High Value')]
print(f'\nPrime Target Segment (High Risk + High Value):')
print(f'  Count         : {len(target):,} customers')
print(f'  Churn rate    : {target["churn"].mean()*100:.1f}%')
print(f'  Avg monthly   : ${target["monthly_charges"].mean():.2f}')
print(f'  Annual revenue at risk: ${target["monthly_charges"].sum()*12:,.0f}')

**Caption:** The top-right cell of the matrix (High Risk × High Value) is the prime target for retention spend. These customers are both the most likely to leave and the most expensive to lose. The scatter plot shows how churned customers (red) cluster in the high-risk, high-charge quadrant.

---
## 3. Retention Strategy Simulation

### Strategy Definitions

| | **Strategy A — Discount Offer** | **Strategy B — Loyalty Perks** |
|-|--------------------------------|---------------------------------|
| **Mechanic** | Offer churning customers 20% bill discount for 6 months | Offer free add-ons: Tech Support + Online Security for 12 months |
| **Cost per customer** | 20% × monthly charge × 6 months | $10/month × 12 months = $120 flat |
| **Target** | All predicted high-risk customers | All predicted high-risk customers |
| **Assumed retention rate** | 45% of contacted customers retained | 35% of contacted customers retained |
| **Rationale** | Discounts are the bluntest instrument — effective but erodes margin | Perks address root cause (lack of sticky services) without directly cutting price |

### Assumptions
- We target the **High Risk** segment identified above (risk_score > 4).
- Retention rate is the probability that a contacted customer does NOT churn after the intervention.
- Revenue after retention = full monthly charge × remaining average tenure (12 months assumed for early churners).
- Strategy A erodes margin: retained customers pay 80% of their bill for 6 months, then full price.
- Strategy B cost is flat $120 per customer regardless of their monthly charge.

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
DISCOUNT_PCT        = 0.20    # 20% bill discount
DISCOUNT_MONTHS     = 6       # for 6 months
RETENTION_RATE_A    = 0.45    # 45% of targeted customers retained by Strategy A

PERK_COST_MONTHLY   = 10.00   # cost of free add-ons to company per month
PERK_MONTHS         = 12      # for 12 months
PERK_COST_FLAT      = PERK_COST_MONTHLY * PERK_MONTHS  # = $120
RETENTION_RATE_B    = 0.35    # 35% of targeted customers retained by Strategy B

RETAINED_AVG_MONTHS = 12      # additional months retained after intervention
MARGIN_RATE         = 0.30

# Target: high risk customers
high_risk   = df_seg[df_seg['risk_tier'] == 'High Risk'].copy()
n_targeted  = len(high_risk)
avg_monthly = high_risk['monthly_charges'].mean()

print(f'High-risk customers targeted : {n_targeted:,}')
print(f'Avg monthly charge (targeted): ${avg_monthly:.2f}')
print(f'Actual churn rate in segment : {high_risk["churn"].mean()*100:.1f}%')

In [ ]:
# ── Strategy A: Discount Offer ────────────────────────────────────────────────
n_retained_A       = int(n_targeted * RETENTION_RATE_A)
n_still_churned_A  = n_targeted - n_retained_A

# Revenue recovered: retained customers pay (1 - discount) for 6 months, then full for 6 months
retained_A = high_risk.head(n_retained_A)  # proxy: take top-n as retained
revenue_discount_period_A = retained_A['monthly_charges'] * (1 - DISCOUNT_PCT) * DISCOUNT_MONTHS
revenue_full_period_A     = retained_A['monthly_charges'] * (RETAINED_AVG_MONTHS - DISCOUNT_MONTHS)
gross_revenue_A           = (revenue_discount_period_A + revenue_full_period_A).sum()

# Cost = the discount given = 20% × monthly × 6 months, for all retained customers
cost_A = (retained_A['monthly_charges'] * DISCOUNT_PCT * DISCOUNT_MONTHS).sum()
net_revenue_A = gross_revenue_A - cost_A
profit_A      = net_revenue_A * MARGIN_RATE

print('=== Strategy A — Discount Offer ===')
print(f'  Customers contacted       : {n_targeted:,}')
print(f'  Customers retained        : {n_retained_A:,}  ({RETENTION_RATE_A*100:.0f}%)')
print(f'  Gross revenue recovered   : ${gross_revenue_A:,.0f}')
print(f'  Cost of discounts given   : ${cost_A:,.0f}')
print(f'  Net revenue recovered     : ${net_revenue_A:,.0f}')
print(f'  Profit recovered (30% mg) : ${profit_A:,.0f}')
print(f'  Cost per retained customer: ${cost_A/n_retained_A:,.0f}')

In [ ]:
# ── Strategy B: Loyalty Perks ─────────────────────────────────────────────────
n_retained_B       = int(n_targeted * RETENTION_RATE_B)
n_still_churned_B  = n_targeted - n_retained_B

retained_B = high_risk.head(n_retained_B)
gross_revenue_B = (retained_B['monthly_charges'] * RETAINED_AVG_MONTHS).sum()

# Cost = flat $120 per contacted customer (perks given to all targeted, not just retained)
cost_B      = n_targeted * PERK_COST_FLAT
net_revenue_B = gross_revenue_B - cost_B
profit_B      = net_revenue_B * MARGIN_RATE

print('=== Strategy B — Loyalty Perks ===')
print(f'  Customers contacted       : {n_targeted:,}')
print(f'  Customers retained        : {n_retained_B:,}  ({RETENTION_RATE_B*100:.0f}%)')
print(f'  Gross revenue recovered   : ${gross_revenue_B:,.0f}')
print(f'  Cost of perks given       : ${cost_B:,.0f}')
print(f'  Net revenue recovered     : ${net_revenue_B:,.0f}')
print(f'  Profit recovered (30% mg) : ${profit_B:,.0f}')
print(f'  Cost per retained customer: ${cost_B/n_retained_B:,.0f}')

In [ ]:
# ── Side-by-side comparison ───────────────────────────────────────────────────
comparison = pd.DataFrame({
    'Metric': [
        'Customers Targeted',
        'Customers Retained',
        'Retention Rate',
        'Gross Revenue Recovered ($)',
        'Total Intervention Cost ($)',
        'Net Revenue Recovered ($)',
        'Profit Recovered ($)',
        'Cost per Retained Customer ($)',
        'Revenue per $1 Spent',
    ],
    'Strategy A — Discount': [
        f'{n_targeted:,}',
        f'{n_retained_A:,}',
        f'{RETENTION_RATE_A*100:.0f}%',
        f'${gross_revenue_A:,.0f}',
        f'${cost_A:,.0f}',
        f'${net_revenue_A:,.0f}',
        f'${profit_A:,.0f}',
        f'${cost_A/n_retained_A:,.0f}',
        f'${gross_revenue_A/cost_A:.2f}',
    ],
    'Strategy B — Perks': [
        f'{n_targeted:,}',
        f'{n_retained_B:,}',
        f'{RETENTION_RATE_B*100:.0f}%',
        f'${gross_revenue_B:,.0f}',
        f'${cost_B:,.0f}',
        f'${net_revenue_B:,.0f}',
        f'${profit_B:,.0f}',
        f'${cost_B/n_retained_B:,.0f}',
        f'${gross_revenue_B/cost_B:.2f}',
    ]
}).set_index('Metric')

print('=== Strategy Comparison Table ===')
comparison

In [ ]:
# ── Visual comparison ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Customers retained
axes[0].bar(['Strategy A\n(Discount)', 'Strategy B\n(Perks)'],
            [n_retained_A, n_retained_B],
            color=[ORANGE, GREEN], edgecolor='white', width=0.45)
axes[0].set_title('Customers Retained')
axes[0].set_ylabel('Count')
for i, v in enumerate([n_retained_A, n_retained_B]):
    axes[0].text(i, v + 5, f'{v:,}', ha='center', fontsize=11, fontweight='bold')

# 2. Net revenue recovered
axes[1].bar(['Strategy A\n(Discount)', 'Strategy B\n(Perks)'],
            [net_revenue_A, net_revenue_B],
            color=[ORANGE, GREEN], edgecolor='white', width=0.45)
axes[1].set_title('Net Revenue Recovered ($)')
axes[1].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
for i, v in enumerate([net_revenue_A, net_revenue_B]):
    axes[1].text(i, v + max(net_revenue_A, net_revenue_B)*0.01,
                 f'${v/1e6:.2f}M', ha='center', fontsize=11, fontweight='bold')

# 3. Cost vs revenue
x = np.arange(2)
width = 0.35
axes[2].bar(x - width/2, [cost_A, cost_B], width, label='Intervention Cost', color=RED, edgecolor='white', alpha=0.85)
axes[2].bar(x + width/2, [profit_A, profit_B], width, label='Profit Recovered', color=GREEN, edgecolor='white', alpha=0.85)
axes[2].set_xticks(x)
axes[2].set_xticklabels(['Strategy A\n(Discount)', 'Strategy B\n(Perks)'])
axes[2].set_title('Cost vs Profit Recovered')
axes[2].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
axes[2].legend()

fig.suptitle('Strategy A vs Strategy B — Simulation Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Break-Even Analysis

How many customers must be retained for each strategy to cover its own cost?

In [ ]:
# Break-even = point where (revenue per retained customer × margin) = cost per retained customer
# Revenue per retained customer = avg_monthly × RETAINED_AVG_MONTHS
avg_monthly_hr = high_risk['monthly_charges'].mean()
revenue_per_retained = avg_monthly_hr * RETAINED_AVG_MONTHS
profit_per_retained  = revenue_per_retained * MARGIN_RATE

# Strategy A: cost scales with customer's monthly charge
avg_discount_cost_per_customer_A = avg_monthly_hr * DISCOUNT_PCT * DISCOUNT_MONTHS
be_customers_A = avg_discount_cost_per_customer_A / profit_per_retained

# Strategy B: flat $120 per contacted customer
# But we pay for ALL targeted, not just retained → cost per targeted = $120
# Break-even: retained × profit_per_retained >= targeted × $120
# n_retained_be_B = (n_targeted × $120) / profit_per_retained
be_customers_B = (n_targeted * PERK_COST_FLAT) / profit_per_retained

print(f'Avg monthly charge (high-risk): ${avg_monthly_hr:.2f}')
print(f'Revenue per retained customer  : ${revenue_per_retained:.2f}  (over {RETAINED_AVG_MONTHS} months)')
print(f'Profit per retained customer   : ${profit_per_retained:.2f}  (30% margin)')
print()
print(f'Strategy A — break-even fraction : {be_customers_A*100:.1f}% of targeted must be retained')
print(f'Strategy A — break-even count    : {int(be_customers_A*n_targeted)+1} customers out of {n_targeted:,}')
print()
print(f'Strategy B — break-even count    : {int(be_customers_B)+1} customers out of {n_targeted:,}')
print(f'Strategy B — break-even fraction : {be_customers_B/n_targeted*100:.1f}% of targeted must be retained')

In [ ]:
# ── Break-even curve ──────────────────────────────────────────────────────────
retention_range = np.arange(0, n_targeted + 1, 10)

# Strategy A: profit = retained × profit_per_retained - retained × avg_discount_cost
profit_curve_A = retention_range * profit_per_retained - retention_range * avg_discount_cost_per_customer_A

# Strategy B: profit = retained × profit_per_retained - n_targeted × $120
profit_curve_B = retention_range * profit_per_retained - (n_targeted * PERK_COST_FLAT)

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(retention_range, profit_curve_A / 1e3, color=ORANGE, lw=2.5, label='Strategy A — Discount')
ax.plot(retention_range, profit_curve_B / 1e3, color=GREEN,  lw=2.5, label='Strategy B — Perks')
ax.axhline(0, color='black', linestyle='--', lw=1.2, label='Break-even ($0)')

# Mark actual simulated outcomes
ax.scatter([n_retained_A], [profit_A/1e3], color=ORANGE, s=100, zorder=5, label=f'Strategy A result ({n_retained_A:,} retained)')
ax.scatter([n_retained_B], [profit_B/1e3], color=GREEN,  s=100, zorder=5, label=f'Strategy B result ({n_retained_B:,} retained)')

ax.set_xlabel('Number of Customers Retained')
ax.set_ylabel('Net Profit ($K)')
ax.set_title('Break-Even Curve — Profit vs Customers Retained per Strategy')
ax.legend(loc='upper left', fontsize=9)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:.0f}K'))
plt.tight_layout()
plt.show()

**Caption:** Strategy A breaks even almost immediately (low per-customer cost). Strategy B has a fixed overhead (you pay for perks for all targeted customers regardless of outcome), so it needs a minimum number of retentions to become profitable — but once past that point, it becomes more efficient per-retained-customer because it doesn't erode margin.

---
## 5. Sensitivity Analysis — What If Assumptions Change?

In [ ]:
# Sweep retention rate from 10% to 70% for both strategies
retention_rates = np.arange(0.05, 0.75, 0.05)

profits_A = []
profits_B = []

for r in retention_rates:
    n_ret = int(n_targeted * r)
    ret_customers = high_risk.head(n_ret)

    # Strategy A
    gr_A = (ret_customers['monthly_charges'] * (1 - DISCOUNT_PCT) * DISCOUNT_MONTHS +
            ret_customers['monthly_charges'] * (RETAINED_AVG_MONTHS - DISCOUNT_MONTHS)).sum()
    c_A  = (ret_customers['monthly_charges'] * DISCOUNT_PCT * DISCOUNT_MONTHS).sum()
    profits_A.append((gr_A - c_A) * MARGIN_RATE)

    # Strategy B
    gr_B = (ret_customers['monthly_charges'] * RETAINED_AVG_MONTHS).sum()
    c_B  = n_targeted * PERK_COST_FLAT
    profits_B.append((gr_B - c_B) * MARGIN_RATE)

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(retention_rates * 100, np.array(profits_A) / 1e3, color=ORANGE, lw=2.5, label='Strategy A — Discount')
ax.plot(retention_rates * 100, np.array(profits_B) / 1e3, color=GREEN,  lw=2.5, label='Strategy B — Perks')
ax.axhline(0, color='black', linestyle='--', lw=1)

# Mark assumed rates
ax.axvline(RETENTION_RATE_A * 100, color=ORANGE, linestyle=':', lw=1.5, label=f'Strategy A assumed rate ({RETENTION_RATE_A*100:.0f}%)')
ax.axvline(RETENTION_RATE_B * 100, color=GREEN,  linestyle=':', lw=1.5, label=f'Strategy B assumed rate ({RETENTION_RATE_B*100:.0f}%)')

ax.set_xlabel('Retention Rate Achieved (%)')
ax.set_ylabel('Net Profit ($K)')
ax.set_title('Sensitivity Analysis — Profit vs Retention Rate Achieved')
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:.0f}K'))
plt.tight_layout()
plt.show()

**Caption:** Strategy A is profitable at almost any retention rate because its cost is proportional to its benefit. Strategy B has a fixed cost floor — it becomes loss-making if the retention rate falls below ~12%. The crossover point shows at what retention rate Strategy B becomes more profitable than A.

---
## 6. Final Segmentation Recommendation

In [ ]:
# Segment summary table
seg_summary = df_seg.groupby(['risk_tier', 'value_tier'], observed=True).agg(
    n_customers   = ('churn', 'count'),
    churn_rate    = ('churn', 'mean'),
    avg_monthly   = ('monthly_charges', 'mean'),
    annual_at_risk= ('monthly_charges', lambda x: x.sum() * 12)
).reset_index()

seg_summary['churn_rate']     = (seg_summary['churn_rate'] * 100).round(1)
seg_summary['avg_monthly']    = seg_summary['avg_monthly'].round(2)
seg_summary['annual_at_risk'] = seg_summary['annual_at_risk'].round(0).astype(int)

seg_summary['recommended_action'] = seg_summary.apply(lambda r: (
    '🔴 Strategy A + B combined — top priority' if r['risk_tier'] == 'High Risk' and r['value_tier'] == 'High Value'
    else '🟠 Strategy A only — cost-sensitive' if r['risk_tier'] == 'High Risk' and r['value_tier'] == 'Mid Value'
    else '🟡 Monitor only — low revenue impact' if r['risk_tier'] == 'High Risk' and r['value_tier'] == 'Low Value'
    else '🔵 Upsell opportunity — low risk' if r['risk_tier'] == 'Low Risk'
    else '⚪ Standard lifecycle management'
), axis=1)

print('=== Segmentation Action Matrix ===')
seg_summary.sort_values(['risk_tier', 'value_tier'], ascending=[False, False])

---
## 7. Retention Recommendation Memo

---

```
MEMO
TO      : Commercial & Customer Success Leadership
FROM    : Data Analytics
SUBJECT : Customer Churn Cost Analysis & Retention Strategy Recommendation
DATE    : [Current Date]
────────────────────────────────────────────────────────────────────────────
```

### The Problem — By the Numbers

26.5% of our customer base churns. That translates to approximately **$2M+ in lost annual recurring revenue**, with an additional **$370K+ in wasted customer acquisition cost** for customers who left before we recovered their onboarding investment. Churned customers pay on average **$14 more per month** than retained customers — meaning we are losing our highest-value accounts at a disproportionate rate.

The churn is not random. It is concentrated in three intersecting segments: **month-to-month contract holders** (43% churn rate), **Fiber Optic subscribers** (42%), and **customers in their first 12 months of tenure** (47%). These segments overlap heavily and represent the core of our churn problem.

### Recommended Strategy — A Two-Track Approach

We simulated two retention interventions against the **high-risk customer pool**. Neither strategy is universally superior — they serve different customer profiles:

**Track 1 — Strategy A (Discount Offer) for Mid-Value High-Risk customers:**  
A 20% bill discount for 6 months. Retains ~45% of targeted customers with a break-even threshold below 5% retention rate — essentially always profitable. Best suited for price-sensitive customers already considering leaving due to cost. Risk: permanently trains customers to expect discounts at renewal time.

**Track 2 — Strategy B (Loyalty Perks) for High-Value High-Risk customers:**  
Free bundling of Tech Support + Online Security for 12 months. Lower retention rate (35%) but addresses the root cause — lack of sticky services — rather than just subsidizing the bill. Once past the ~12% break-even threshold, this strategy generates higher per-customer profit than discounting and does not erode pricing integrity.

**For the prime segment (High Risk × High Value):** deploy both simultaneously. Lead with perks; offer a discount only if the customer escalates or declines the perk offer.

### Three Immediate Actions

1. **First 90 days:** Deploy a proactive outreach program to the ~650 High Risk × High Value customers. These customers represent the highest revenue-at-risk concentration. Assign dedicated account managers or a retention pod for this cohort specifically.

2. **Contract conversion campaign:** Month-to-month customers who have been active for 6+ months are the ideal upgrade target. Offer a first-year annual contract at a 10% discount — cheaper than a churn event and eliminates the monthly exit option. A 10% contract conversion rate on this group would prevent an estimated 200+ annual churns.

3. **Fiber onboarding redesign:** Fiber Optic users churn at 2× the rate of DSL users despite paying more. This is a service quality and expectation gap, not purely a price issue. A dedicated fiber onboarding call at day 30, with proactive Tech Support enrollment, would address the spike in first-year fiber churn at near-zero intervention cost.

### Limitations of This Analysis

The simulation uses assumed retention rates (45% for discounts, 35% for perks) sourced from industry benchmarks. These should be validated through an A/B test before full deployment. The financial model also uses a static 30% margin assumption — actual margin varies by plan tier. The model should be re-run with actual cost-of-goods data once available.

---
*End of Memo*